# Vision-Language Models — CLIP from scratch in NumPy

**Course:** [Computer Vision · Vision-Language Models](https://ml-viz-ruby.vercel.app/courses/computer-vision/04-vision-language-models)

We build a tiny toy CLIP in pure NumPy. Two encoders (image and text) project into a shared embedding space, we train the symmetric contrastive loss, and then verify two consequences: a diagonally-dominant similarity matrix on a held-out batch, and zero-shot classification from text prototypes.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

rng = np.random.default_rng(7)


## 1. Build two toy modalities with shared class structure

Real images and text live in totally different surface spaces, but a CLIP-style model has to recover the fact that an image of a cat and a sentence about cats share an underlying *class*. We simulate this by drawing all samples from a low-rank class-conditional structure in a hidden 4-D space, then projecting it into two **different** observation spaces: 64-D for "images" and 32-D for "text".


In [ ]:
N_CLASSES = 4
PER_CLASS = 16  # samples per class, per modality
HIDDEN = 4
D_IMG = 64
D_TXT = 32

# A class prototype in hidden space — the *shared* signal across modalities.
class_protos = rng.normal(size=(N_CLASSES, HIDDEN))

# Per-sample hidden code: class prototype + tiny noise.
def sample_hidden(n_samples_per_class, noise=0.3):
    hidden = []
    labels = []
    for c in range(N_CLASSES):
        block = class_protos[c] + noise * rng.normal(size=(n_samples_per_class, HIDDEN))
        hidden.append(block)
        labels.extend([c] * n_samples_per_class)
    return np.concatenate(hidden, axis=0), np.array(labels)

# Two *independent* random projections into the two surface spaces.
P_img = rng.normal(size=(HIDDEN, D_IMG)) / np.sqrt(HIDDEN)
P_txt = rng.normal(size=(HIDDEN, D_TXT)) / np.sqrt(HIDDEN)

# Build the matched pairs: the i-th image and i-th text share the same hidden code.
h_train, y_train = sample_hidden(PER_CLASS, noise=0.3)
image_features_train = h_train @ P_img + 0.4 * rng.normal(size=(len(h_train), D_IMG))
text_features_train  = h_train @ P_txt + 0.4 * rng.normal(size=(len(h_train), D_TXT))

print('image_features_train:', image_features_train.shape)
print('text_features_train :', text_features_train.shape)
print('labels              :', y_train[:10], '…')


## 2. Learnable encoders — one linear layer each into a shared $d$-dim space

This is the smallest possible CLIP. Each encoder is one matrix, the output is $\ell_2$-normalised, and the similarity is a dot product on unit vectors.


In [ ]:
D = 16  # shared embedding dim

# Init encoder matrices.
W_img = rng.normal(size=(D_IMG, D)) / np.sqrt(D_IMG)
W_txt = rng.normal(size=(D_TXT, D)) / np.sqrt(D_TXT)

def l2_normalise(X, eps=1e-9):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def encode_img(X, W=W_img):
    return l2_normalise(X @ W)

def encode_txt(X, W=W_txt):
    return l2_normalise(X @ W)

# Sanity check: every row has unit norm.
z_img = encode_img(image_features_train)
z_txt = encode_txt(text_features_train)
print('||z_img||:', np.linalg.norm(z_img, axis=1)[:5])
print('||z_txt||:', np.linalg.norm(z_txt, axis=1)[:5])


## 3. The symmetric CLIP loss + a hand-rolled gradient

Given a batch of $B$ matched pairs and a temperature $\tau$:

$$
S_{ij} = z^{\text{img}}_i \cdot z^{\text{txt}}_j / \tau
$$

$$
\mathcal{L} = \tfrac{1}{2}(\mathrm{CE}(\text{softmax}_\text{row}(S),\ I) + \mathrm{CE}(\text{softmax}_\text{col}(S),\ I))
$$

We compute the loss in NumPy and use a finite-difference numerical gradient — slow but transparent. Good enough to actually train the toy.


In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def clip_loss(z_img, z_txt, tau=0.1):
    """Symmetric CLIP loss on already-normalised embeddings."""
    B = z_img.shape[0]
    S = z_img @ z_txt.T / tau               # B x B logits
    p_rows = softmax(S, axis=1)             # P(text | image)
    p_cols = softmax(S, axis=0)             # P(image | text)
    diag_idx = np.arange(B)
    loss_i2t = -np.log(p_rows[diag_idx, diag_idx] + 1e-12).mean()
    loss_t2i = -np.log(p_cols[diag_idx, diag_idx] + 1e-12).mean()
    return 0.5 * (loss_i2t + loss_t2i)

# Quick sanity check: at random init the loss should be ~ log B.
B = image_features_train.shape[0]
init_loss = clip_loss(encode_img(image_features_train), encode_txt(text_features_train), tau=0.1)
print(f'B = {B}, log B = {np.log(B):.3f}, init loss = {init_loss:.3f}')


## 4. Train with finite-difference gradient descent

For pedagogy we differentiate the loss with a tiny finite-difference probe rather than coding the analytic gradient by hand. This is slow but correct, and the toy is small enough that ~500 steps finish in a few seconds.


In [ ]:
def loss_for(W_img_, W_txt_, tau=0.1):
    z_i = l2_normalise(image_features_train @ W_img_)
    z_t = l2_normalise(text_features_train @ W_txt_)
    return clip_loss(z_i, z_t, tau=tau)

def numerical_grad(W, fn, eps=1e-3):
    g = np.zeros_like(W)
    base = fn(W)
    flat = W.reshape(-1)
    grad_flat = g.reshape(-1)
    # Probe in a small random subspace to keep this fast (proper SGD-of-gradient
    # would do the whole matrix; the random subset is cheaper and noisy but
    # works for the toy.)
    idx = rng.choice(flat.size, size=min(96, flat.size), replace=False)
    for k in idx:
        flat[k] += eps
        plus = fn(W)
        flat[k] -= 2 * eps
        minus = fn(W)
        flat[k] += eps  # restore
        grad_flat[k] = (plus - minus) / (2 * eps)
    return g

losses = []
lr = 0.6
TAU = 0.1
W_img_train = W_img.copy()
W_txt_train = W_txt.copy()
for step in range(500):
    g_img = numerical_grad(W_img_train, lambda W: loss_for(W, W_txt_train, tau=TAU))
    g_txt = numerical_grad(W_txt_train, lambda W: loss_for(W_img_train, W, tau=TAU))
    W_img_train -= lr * g_img
    W_txt_train -= lr * g_txt
    losses.append(loss_for(W_img_train, W_txt_train, tau=TAU))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color='#6366f1', linewidth=2)
ax.axhline(np.log(B), color='#f97316', linestyle=':', alpha=0.7, label=f'log B = {np.log(B):.2f} (chance)')
ax.set_xlabel('step')
ax.set_ylabel('symmetric CLIP loss')
ax.set_title('Training the toy CLIP (NumPy)', fontsize=12)
ax.legend(facecolor='#1a1d27', edgecolor='#2a2d3a')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'final loss: {losses[-1]:.3f}  (vs random init {init_loss:.3f})')


## 5. The similarity matrix on the trained batch — the diagonal should light up

The CLIP loss directly optimises the diagonal of $S$ on each training batch. We visualise that diagonal lighting up on a small slice of the training set. (Generalising to truly held-out *pairwise* retrieval would take orders of magnitude more data and capacity than a single linear layer — the toy here is just to make the training objective concrete. Section 6 shows that **class-level** generalisation already works with these toy encoders, which is the more useful demo.)

In [ ]:
# Visualise a slice of the trained-batch similarity matrix.
B_show = 8
slice_idx = rng.choice(image_features_train.shape[0], size=B_show, replace=False)

z_img_slice = l2_normalise(image_features_train[slice_idx] @ W_img_train)
z_txt_slice = l2_normalise(text_features_train[slice_idx] @ W_txt_train)
S_slice = z_img_slice @ z_txt_slice.T

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(S_slice, cmap='viridis')
ax.set_xlabel('text index')
ax.set_ylabel('image index')
ax.set_title('Trained-batch cosine-similarity matrix\n(diagonal positives should dominate)', fontsize=11)
plt.colorbar(im, ax=ax, label='cos(z_img, z_txt)')
plt.tight_layout()
plt.show()

# In-batch retrieval accuracy — what the contrastive loss directly optimises.
preds = S_slice.argmax(axis=1)
acc = (preds == np.arange(B_show)).mean()
print(f'in-batch image -> text top-1 retrieval accuracy: {acc:.2%}')

## 6. Zero-shot classification via class prototypes

The whole point of CLIP at inference time: build a *text* embedding per class, then classify any new image by nearest text prototype. Here we average the trained text embeddings within each class to form prototypes.


In [ ]:
# Build a class prototype in shared space by averaging the trained text
# embeddings within each class on the training set.
z_txt_train = l2_normalise(text_features_train @ W_txt_train)
prototypes = np.zeros((N_CLASSES, D))
for c in range(N_CLASSES):
    mask = y_train == c
    proto = z_txt_train[mask].mean(axis=0)
    prototypes[c] = proto / (np.linalg.norm(proto) + 1e-9)

# Fresh held-out images we have *never* trained on.
h_test, y_test = sample_hidden(8, noise=0.3)
img_test = h_test @ P_img + 0.4 * rng.normal(size=(len(h_test), D_IMG))

# Encode held-out images, classify by nearest *text-derived* class prototype.
z_img_test_proto = l2_normalise(img_test @ W_img_train)
scores = z_img_test_proto @ prototypes.T   # (N_test, N_CLASSES)
preds = scores.argmax(axis=1)
acc = (preds == y_test).mean()
print(f'zero-shot classification accuracy on held-out images: {acc:.2%}')
print(f'(chance = {100 / N_CLASSES:.1f}%)')

## 7. Averaging multiple "prompt templates" beats one template

We can't conjure real LLM tokens here, but we *can* simulate the effect of using multiple prompt templates by adding small independent noise to a prompt embedding and then averaging. This is structurally identical to averaging `"a photo of a cat"`, `"a picture of a cat"`, `"a close-up of a cat"`, ... at inference time in real CLIP.


In [ ]:
TEMPLATE_NOISE = 0.7  # the magnitude of template-specific noise
N_TEMPLATES = 4

def proto_single_template():
    """Build one class prototype from one noisy text embedding per class."""
    proto = np.zeros((N_CLASSES, D))
    for c in range(N_CLASSES):
        mask = y_train == c
        idx = rng.choice(mask.sum())
        z = z_txt_train[mask][idx] + TEMPLATE_NOISE * rng.normal(size=D)
        z = z / (np.linalg.norm(z) + 1e-9)
        proto[c] = z
    return proto

def proto_averaged_templates():
    """Average N_TEMPLATES noisy unit-norm embeddings per class, then re-normalise."""
    proto = np.zeros((N_CLASSES, D))
    for c in range(N_CLASSES):
        mask = y_train == c
        accum = np.zeros(D)
        for _ in range(N_TEMPLATES):
            idx = rng.choice(mask.sum())
            z = z_txt_train[mask][idx] + TEMPLATE_NOISE * rng.normal(size=D)
            z = z / (np.linalg.norm(z) + 1e-9)
            accum += z
        accum /= N_TEMPLATES
        proto[c] = accum / (np.linalg.norm(accum) + 1e-9)
    return proto

def evaluate(proto):
    scores = z_img_test_proto @ proto.T
    return (scores.argmax(axis=1) == y_test).mean()

# Run each strategy multiple times to wash out the random template draw.
runs = 20
acc_single = np.mean([evaluate(proto_single_template()) for _ in range(runs)])
acc_avg = np.mean([evaluate(proto_averaged_templates()) for _ in range(runs)])
print(f'single template:  {acc_single:.2%}')
print(f'{N_TEMPLATES}-template avg: {acc_avg:.2%}   (averaging cancels template-specific noise)')


## ✏️ Your turn

### Implement the symmetric CLIP loss yourself

Fill in `clip_loss_yours(z_img, z_txt, tau)`. Inputs are already $\ell_2$-normalised. Compute the $B \times B$ similarity matrix, the symmetric cross-entropy (rows = "image → which text", columns = "text → which image"), and return the average of the two terms. Check it matches the reference at random initialisation: for batch size $B$, $\mathcal{L}_0 \approx \log B$.


In [ ]:
def clip_loss_yours(z_img, z_txt, tau=0.1):
    """
    Symmetric CLIP loss. z_img and z_txt are already L2-normalised (B, D).
    Diagonal positives. Return the average of the row-CE and col-CE.
    """
    # TODO(you):
    # 1. Build S = z_img @ z_txt.T / tau   (shape B x B)
    # 2. Compute softmax along rows (P(text | image)) and columns (P(image | text))
    # 3. Pick the diagonal probabilities, take -log, mean.
    # 4. Return 0.5 * (row_loss + col_loss).
    pass

# Reference: symmetric loss at random init must be approximately log(B).
B_check = 32
rng_check = np.random.default_rng(123)
z_i = l2_normalise(rng_check.normal(size=(B_check, 8)))
z_t = l2_normalise(rng_check.normal(size=(B_check, 8)))
yours = clip_loss_yours(z_i, z_t, tau=1.0)
ref = clip_loss(z_i, z_t, tau=1.0)


In [ ]:
# Sanity assertions: same value as the reference, AND ~ log B at random init.
yours = clip_loss_yours(z_i, z_t, tau=1.0)
assert yours is not None, 'You need to return a value from clip_loss_yours'
assert np.isclose(yours, ref, atol=1e-6), f'Should match reference: got {yours}, expected {ref}'
assert abs(yours - np.log(B_check)) < 0.5, (
    f'Random-init loss should be near log B = {np.log(B_check):.3f}, got {yours:.3f}'
)
print(f'✅ Your CLIP loss = {yours:.3f}, matches reference {ref:.3f}, and is close to log B = {np.log(B_check):.3f}')


<details>
<summary>💡 Show solution</summary>

```python
def clip_loss_yours(z_img, z_txt, tau=0.1):
    B = z_img.shape[0]
    S = z_img @ z_txt.T / tau
    # Row softmax: P(text_j | image_i)
    S_row = S - S.max(axis=1, keepdims=True)
    p_row = np.exp(S_row) / np.exp(S_row).sum(axis=1, keepdims=True)
    # Column softmax: P(image_i | text_j)
    S_col = S - S.max(axis=0, keepdims=True)
    p_col = np.exp(S_col) / np.exp(S_col).sum(axis=0, keepdims=True)
    diag = np.arange(B)
    loss_i2t = -np.log(p_row[diag, diag] + 1e-12).mean()
    loss_t2i = -np.log(p_col[diag, diag] + 1e-12).mean()
    return 0.5 * (loss_i2t + loss_t2i)
```

At random initialisation each row's softmax is roughly uniform, so the diagonal probability is $\approx 1/B$, the per-direction cross-entropy is $\approx \log B$, and the symmetric average is also $\log B$. This is the floor: training reduces it.
</details>
